# LLM Judge Post-processing — 3 Criteria Version

Notebook này post-process `judge_results.csv` / `judge_results.jsonl` chỉ bằng 3 tiêu chí:

1. `meaning_score`
2. `naturalness_score`
3. `social_style_score`

Không dùng:

- `dataset_usefulness_score`
- `ai_artifact_score`
- `language_consistency_score`
- `overall_quality_score`

Tổng điểm tối đa là:

```text
3 tiêu chí x 5 điểm = 15 điểm
```

Decision rule:

```text
total_score <= 7       → drop
8 <= total_score <= 12 → review
total_score >= 13      → keep
```


## 1. Import libraries


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)


## 2. Configuration


In [11]:

# Change these paths to match your environment.
INPUT_PATH = "D:\IE403\ViAIGS\outputs\judge_results.csv"
OUTPUT_DIR = "D:\IE403\ViAIGS\outputs\llm_judge_3criteria"

CRITERIA_COLUMNS = [
    "meaning_score",
    "naturalness_score",
    "social_style_score",
]

MAX_TOTAL_SCORE = 15

DROP_MAX_SCORE = 6
REVIEW_MIN_SCORE = 7
REVIEW_MAX_SCORE = 11
KEEP_MIN_SCORE = 12

input_path = Path(INPUT_PATH)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

print("Input path:", input_path)
print("Output dir:", output_dir)
print("Criteria:", CRITERIA_COLUMNS)


Input path: D:\IE403\ViAIGS\outputs\judge_results.csv
Output dir: D:\IE403\ViAIGS\outputs\llm_judge_3criteria
Criteria: ['meaning_score', 'naturalness_score', 'social_style_score']


## 3. Load results


In [12]:

if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path}")

if input_path.suffix.lower() == ".jsonl":
    df = pd.read_json(input_path, lines=True)
else:
    df = pd.read_csv(input_path, encoding="utf-8-sig")

if "judge_item_id" not in df.columns:
    raise ValueError("Input file must contain judge_item_id column.")

print(f"Loaded rows: {len(df):,}")
display(df.head())


Loaded rows: 9,001


,judge_item_id,original_id,candidate_id,generator,source,original_text,candidate_text,surface_similarity,empty_original,empty_candidate,ai_artifact_flag,encoding_noise_flag,too_short_flag,too_similar_flag,rule_predecision,length,candidate_length,paraphrase_round,generation_method,bleu_score,bertscore,crossencoder_score,ld,edit_similarity,meaning_score,naturalness_score,social_style_score,language_consistency_score,ai_artifact_score,dataset_usefulness_score,overall_quality_score,decision,final_decision,reason,raw_judge_response,judge_error
0,aya::human_013660_para_r1,human_013660,human_013660_para_r1,aya,fb,Nguyễn Bích Như học tập đi,"Nâng niu mơ ước, Nguyễn Bích Như hãy cố gắng học tập thật tốt!",0.5455,False,False,False,True,False,False,drop,6.0,14,1,paraphrase,0.0155,0.7080,0.4910,56,0.2911,NaN,NaN,NaN,NaN,NaN,NaN,NaN,drop,drop,Dropped by rule-based precheck.,NaN,NaN
1,aya::human_025981_para_r3,human_025981,human_025981_para_r3,aya,ytb,Trên đất nước nghèo đói của Việt Nam có mấy người như anh Vũ. Người toàn tâm toàn ý vì sự phát triến đất nước. Nhiều người có chúc quyền cao sẽ đồ kỵ Anh. Vì anh không...,"Trên đất nước Việt Nam nghèo khó, có những cá nhân như anh Vũ, những con người cống hiến hết mình cho sự phát triển của đất nước. Nhiều người nắm quyền lực sẽ e ngại anh vì sự ...",0.2016,False,False,False,True,False,False,drop,72.0,100,3,paraphrase,0.0599,0.7111,0.9977,299,0.3385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,drop,drop,Dropped by rule-based precheck.,NaN,NaN
2,aya::human_028119_para_r2,human_028119,human_028119_para_r2,aya,ytb,"Mình giới thiệu 1 cuốn sách trinh thám kinh điển của 1 tác giả Việt Nam, xuất bản năm 2000. Sách được chuyển thể thành phim, và gây tiếng vang lớn trong những năm đầu thế kỷ 21...","Cuốn sách trinh thám kinh điển, ra mắt vào năm 2000, của một tác giả người Việt Nam đã thu hút sự chú ý lớn trong những năm đầu thập kỷ 21 khi xã hội Việt Nam vẫn còn khá chưa ...",0.3071,False,False,False,True,False,False,drop,137.0,117,2,paraphrase,0.1356,0.7762,0.9997,402,0.3537,NaN,NaN,NaN,NaN,NaN,NaN,NaN,drop,drop,Dropped by rule-based precheck.,NaN,NaN
3,aya::human_016713_para_r3,human_016713,human_016713_para_r3,aya,ytb,Mua dép thì phải làm như thế nào ạ?,"Khi mua dép, bạn cần lưu ý những điều sau đây nhé!",0.3765,False,False,False,True,False,False,drop,9.0,12,3,paraphrase,0.0155,0.7191,0.9709,35,0.3000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,drop,drop,Dropped by rule-based precheck.,NaN,NaN
4,aya::human_014594_para_r3,human_014594,human_014594_para_r3,aya,fb,Thảo Uyên Nguyễn,Thảo Uyên Nguyễn vừa bày tỏ tâm trạng mới nhất của cô ấy trên mạng xã hội.,0.3556,False,False,False,True,False,False,drop,3.0,17,3,paraphrase,0.0155,0.7587,0.6744,58,0.2162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,drop,drop,Dropped by rule-based precheck.,NaN,NaN


## 4. Prepare score columns


In [13]:

missing_cols = [c for c in CRITERIA_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing criteria columns: {missing_cols}")

for col in CRITERIA_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["missing_3criteria_score_count"] = df[CRITERIA_COLUMNS].isna().sum(axis=1)

# Missing scores are set to 0 to avoid accidentally keeping incomplete judge results.
df[CRITERIA_COLUMNS] = df[CRITERIA_COLUMNS].fillna(0)

# Safety clipping.
for col in CRITERIA_COLUMNS:
    df[col] = df[col].clip(lower=0, upper=5)

display(df[CRITERIA_COLUMNS].describe().T)
display(
    df["missing_3criteria_score_count"]
    .value_counts()
    .sort_index()
    .rename_axis("missing_3criteria_score_count")
    .reset_index(name="rows")
)


,count,mean,std,min,25%,50%,75%,max
meaning_score,9001.0,2.192645,2.262200,0.0,0.0,1.0,5.0,5.0
naturalness_score,9001.0,2.167093,2.122648,0.0,0.0,2.0,4.0,5.0
social_style_score,9001.0,2.541051,2.438376,0.0,0.0,3.0,5.0,5.0


,missing_3criteria_score_count,rows
0,0,4761
1,3,4240


## 5. Compute 3-criteria score


In [14]:

df["quality_3criteria_total_score"] = df[CRITERIA_COLUMNS].sum(axis=1)
df["quality_3criteria_avg_score"] = df[CRITERIA_COLUMNS].mean(axis=1)

display(df["quality_3criteria_total_score"].describe())

score_bins = pd.cut(
    df["quality_3criteria_total_score"],
    bins=[-0.1, DROP_MAX_SCORE, REVIEW_MAX_SCORE, MAX_TOTAL_SCORE],
    labels=[
        f"0-{DROP_MAX_SCORE}",
        f"{REVIEW_MIN_SCORE}-{REVIEW_MAX_SCORE}",
        f"{KEEP_MIN_SCORE}-{MAX_TOTAL_SCORE}",
    ],
    include_lowest=True,
)

display(
    score_bins
    .value_counts()
    .sort_index()
    .rename_axis("score_range")
    .reset_index(name="rows")
)


count    9001.000000
mean        6.900789
std         6.727542
min         0.000000
25%         0.000000
50%         8.000000
75%        14.000000
max        15.000000
Name: quality_3criteria_total_score, dtype: float64

,score_range,rows
0,0-6,4479
1,7-11,350
2,12-15,4172


## 6. Apply decision rule


In [15]:

def three_criteria_decision(total_score):
    if pd.isna(total_score):
        return "drop"
    if total_score <= DROP_MAX_SCORE:
        return "drop"
    if REVIEW_MIN_SCORE <= total_score <= REVIEW_MAX_SCORE:
        return "review"
    if total_score >= KEEP_MIN_SCORE:
        return "keep"
    return "review"


def three_criteria_reason(total_score):
    if pd.isna(total_score):
        return "missing_total_score_drop"
    if total_score <= DROP_MAX_SCORE:
        return f"total_score_{total_score:.0f}_le_{DROP_MAX_SCORE}_drop"
    if REVIEW_MIN_SCORE <= total_score <= REVIEW_MAX_SCORE:
        return f"total_score_{total_score:.0f}_between_{REVIEW_MIN_SCORE}_{REVIEW_MAX_SCORE}_review"
    if total_score >= KEEP_MIN_SCORE:
        return f"total_score_{total_score:.0f}_ge_{KEEP_MIN_SCORE}_keep"
    return "undefined_score_range_review"


df["three_criteria_decision"] = df["quality_3criteria_total_score"].apply(three_criteria_decision)
df["three_criteria_reason"] = df["quality_3criteria_total_score"].apply(three_criteria_reason)

print("3-criteria decision distribution:")
display(
    df["three_criteria_decision"]
    .value_counts(dropna=False)
    .rename_axis("three_criteria_decision")
    .reset_index(name="rows")
)

if "final_decision" in df.columns:
    print("Original final_decision vs three_criteria_decision:")
    display(pd.crosstab(df["final_decision"], df["three_criteria_decision"], margins=True))


3-criteria decision distribution:


,three_criteria_decision,rows
0,drop,4479
1,keep,4172
2,review,350


Original final_decision vs three_criteria_decision:


three_criteria_decision,drop,keep,review,All
final_decision,,,,
drop,4430,4030,348,8808
keep,0,138,0,138
review,49,4,2,55
All,4479,4172,350,9001


## 7. Compare rescued rows


In [16]:

if "final_decision" in df.columns:
    rescued = df[
        (df["final_decision"].astype(str).str.lower() == "drop")
        & (df["three_criteria_decision"].isin(["review", "keep"]))
    ].copy()

    print(f"Rows changed from old drop to review/keep: {len(rescued):,}")

    inspect_cols = [
        "judge_item_id",
        "generator",
        "source",
        "original_text",
        "candidate_text",
        "quality_3criteria_total_score",
        "quality_3criteria_avg_score",
        "three_criteria_decision",
        "meaning_score",
        "naturalness_score",
        "social_style_score",
        "decision",
        "final_decision",
        "reason",
    ]
    inspect_cols = [c for c in inspect_cols if c in rescued.columns]
    display(rescued[inspect_cols].head(20))
else:
    print("No final_decision column found.")


Rows changed from old drop to review/keep: 4,378


,judge_item_id,generator,source,original_text,candidate_text,quality_3criteria_total_score,quality_3criteria_avg_score,three_criteria_decision,meaning_score,naturalness_score,social_style_score,decision,final_decision,reason
4192,aya::human_011710_para_r2,aya,fb,chả mất gì chỉ mất tiềnnn,"chưa mất gì, chỉ mất tiền thôi",15.0,5.000000,keep,5.0,5.0,5.0,review,drop,"obvious AI artifact in the phrase 'chưa mất gì, chỉ mất tiền thôi' with a slight deviation from the original meaning"
4193,aya::human_011834_para_r1,aya,fb,Cố lên những người con đất việt chúc chuyến đi thành công và tốt đẹp,Chúc các bạn con đất Việt một chuyến đi an toàn và thú vị!,14.0,4.666667,keep,5.0,4.0,5.0,review,drop,"Candidate preserves the original meaning and intent, sounds natural and fits social media style, but contains an obvious AI artifact (template-like phrasing)."
4194,aya::human_030950_para_r3,aya,ytb,Mong sao hết dịch giống vậy thì đẹp,"Nếu có thể, tôi ước đại dịch sẽ kết thúc và cuộc sống sẽ trở lại nhịp điệu tự nhiên thường ngày.",7.0,2.333333,review,2.0,3.0,2.0,review,drop,Candidate preserves some meaning but is overly formal and contains an AI artifact.
4195,aya::human_016887_para_r2,aya,ytb,Anh tôi bịp ghê 🌚,Em trai mình thật là nghịch ngợm nhỉ 😎,15.0,5.000000,keep,5.0,5.0,5.0,review,drop,Candidate contains an obvious AI artifact (template phrase) but preserves the original meaning and social style.
4196,aya::human_031646_para_r2,aya,fb,"Lại va phải ""nó"" rồi :) #HwangLong #welax","Thách thức ""nó"" luôn luôn thôi :) #HwangLong #viux",14.0,4.666667,keep,5.0,4.0,5.0,review,drop,Candidate has slight AI artifact (template-like phrasing) but preserves meaning and social style. Minor inconsistency in language use and slight naturalness issues.
4197,aya::human_031436_para_r3,aya,fb,Vua của lười =))) #HwangLong #welax,Vua lười biếng =)) #HwangLong #tài_hiệu,14.0,4.666667,keep,5.0,4.0,5.0,review,drop,Obvious AI artifact in the #tài_hiệu tag and slight deviation in meaning from the original text
4198,aya::human_017396_para_r2,aya,ytb,Mèo thông minh như độ lươn lẹo đẩy nhau,"Những chú mèo thông minh như những chú rùa lẹ lẹo, luôn tìm cách giúp đỡ lẫn nhau.",12.0,4.000000,keep,3.0,4.0,5.0,review,drop,The candidate preserves the original meaning but has minor AI artifacts and is slightly unnatural in style.
4199,aya::human_000419_para_r3,aya,voz,"Ko sao, it việt nam không bị ảnh hưởng gì đâu Nó chừa mình ra, cu thể là ko bao giờ đuổi dev","Thật bất ngờ, Việt Nam vẫn không bị ảnh hưởng. Đất nước vẫn giữ nguyên bản sắc riêng và có thể sẽ không bao giờ theo kịp những nước phát triển.",12.0,4.000000,keep,3.0,4.0,5.0,review,drop,Candidate preserves meaning but has minor AI artifacts and is slightly unnatural
4200,aya::human_013719_para_r2,aya,fb,Dân trí cao nên khổ vậy đấy Khéo bỏ nấu cơm tối cho gia đình ra hóng :v,Có học thức nhưng cuộc sống vẫn khổ thế cơ! Thỉnh thoảng nấu cơm tối cho cả nhà cùng thưởng thức thật tuyệt! 😜,14.0,4.666667,keep,5.0,4.0,5.0,review,drop,"Candidate preserves the original meaning and intent, sounds natural and fits social media style, but contains an obvious AI artifact (template phrase) and low dataset usefulnes..."
4201,aya::human_019285_para_r2,aya,ytb,Mình rất thích những món này,Mình thực sự yêu thích những món ăn tuyệt vời này!,15.0,5.000000,keep,5.0,5.0,5.0,review,drop,"Candidate has slight AI artifact (template-like phrasing) but preserves meaning and naturalness, fits social style, and is consistent in language."


## 8. Export outputs


In [17]:

full_output = output_dir / "final_3criteria_results.csv"
accepted_output = output_dir / "accepted_3criteria.csv"
review_output = output_dir / "review_3criteria.csv"
dropped_output = output_dir / "dropped_3criteria.csv"

df.to_csv(full_output, index=False, encoding="utf-8-sig")

accepted = df[df["three_criteria_decision"] == "keep"].copy()
review = df[df["three_criteria_decision"] == "review"].copy()
dropped = df[df["three_criteria_decision"] == "drop"].copy()

accepted.to_csv(accepted_output, index=False, encoding="utf-8-sig")
review.to_csv(review_output, index=False, encoding="utf-8-sig")
dropped.to_csv(dropped_output, index=False, encoding="utf-8-sig")

print(f"Full results: {full_output}")
print(f"Accepted: {accepted_output}")
print(f"Review: {review_output}")
print(f"Dropped: {dropped_output}")

print()
print(f"Accepted rows: {len(accepted):,}")
print(f"Review rows:   {len(review):,}")
print(f"Dropped rows:  {len(dropped):,}")

display(accepted.head())


Full results: D:\IE403\ViAIGS\outputs\llm_judge_3criteria\final_3criteria_results.csv
Accepted: D:\IE403\ViAIGS\outputs\llm_judge_3criteria\accepted_3criteria.csv
Review: D:\IE403\ViAIGS\outputs\llm_judge_3criteria\review_3criteria.csv
Dropped: D:\IE403\ViAIGS\outputs\llm_judge_3criteria\dropped_3criteria.csv

Accepted rows: 4,172
Review rows:   350
Dropped rows:  4,479


,judge_item_id,original_id,candidate_id,generator,source,original_text,candidate_text,surface_similarity,empty_original,empty_candidate,ai_artifact_flag,encoding_noise_flag,too_short_flag,too_similar_flag,rule_predecision,length,candidate_length,paraphrase_round,generation_method,bleu_score,bertscore,crossencoder_score,ld,edit_similarity,meaning_score,naturalness_score,social_style_score,language_consistency_score,ai_artifact_score,dataset_usefulness_score,overall_quality_score,decision,final_decision,reason,raw_judge_response,judge_error,missing_3criteria_score_count,quality_3criteria_total_score,quality_3criteria_avg_score,three_criteria_decision,three_criteria_reason
4191,aya::human_025003_para_r1,human_025003,human_025003_para_r1,aya,ytb,cách diễn của hl làm tui cười muốn chết,cái cách diễn của Hl khiến tui cười muốn điên,0.7381,False,False,False,False,False,False,judge,9.0,10,1,paraphrase,0.0188,0.7034,0.9560,42,0.4000,5.0,5.0,5.0,5.0,5.0,5.0,5.0,keep,keep,"Candidate preserves the original meaning, sounds natural and fits social media style, uses consistent language, and is free of AI artifacts.","{\n ""meaning_score"": 5,\n ""naturalness_score"": 5,\n ""social_style_score"": 5,\n ""language_consistency_score"": 5,\n ""ai_artifact_score"": 5,\n ""dataset_usefulness_score"": 5,...",NaN,0,15.0,5.000000,keep,total_score_15_ge_12_keep
4192,aya::human_011710_para_r2,human_011710,human_011710_para_r2,aya,fb,chả mất gì chỉ mất tiềnnn,"chưa mất gì, chỉ mất tiền thôi",0.8000,False,False,False,False,False,False,judge,6.0,7,2,paraphrase,0.0631,0.7722,0.9609,12,0.6471,5.0,5.0,5.0,5.0,1.0,5.0,5.0,review,drop,"obvious AI artifact in the phrase 'chưa mất gì, chỉ mất tiền thôi' with a slight deviation from the original meaning","{\n ""meaning_score"": 5,\n ""naturalness_score"": 5,\n ""social_style_score"": 5,\n ""language_consistency_score"": 5,\n ""ai_artifact_score"": 1,\n ""dataset_usefulness_score"": 5,...",NaN,0,15.0,5.000000,keep,total_score_15_ge_12_keep
4193,aya::human_011834_para_r1,human_011834,human_011834_para_r1,aya,fb,Cố lên những người con đất việt chúc chuyến đi thành công và tốt đẹp,Chúc các bạn con đất Việt một chuyến đi an toàn và thú vị!,0.5714,False,False,False,False,False,False,judge,15.0,14,1,paraphrase,0.0433,0.7553,0.2880,50,0.2647,5.0,4.0,5.0,5.0,1.0,5.0,5.0,review,drop,"Candidate preserves the original meaning and intent, sounds natural and fits social media style, but contains an obvious AI artifact (template-like phrasing).","{\n ""meaning_score"": 5,\n ""naturalness_score"": 4,\n ""social_style_score"": 5,\n ""language_consistency_score"": 5,\n ""ai_artifact_score"": 1,\n ""dataset_usefulness_score"": 5,...",NaN,0,14.0,4.666667,keep,total_score_14_ge_12_keep
4195,aya::human_016887_para_r2,human_016887,human_016887_para_r2,aya,ytb,Anh tôi bịp ghê 🌚,Em trai mình thật là nghịch ngợm nhỉ 😎,0.3273,False,False,False,False,False,False,judge,5.0,9,2,paraphrase,0.0000,0.7395,0.5434,39,0.2500,5.0,5.0,5.0,5.0,1.0,5.0,5.0,review,drop,Candidate contains an obvious AI artifact (template phrase) but preserves the original meaning and social style.,"{\n ""meaning_score"": 5,\n ""naturalness_score"": 5,\n ""social_style_score"": 5,\n ""language_consistency_score"": 5,\n ""ai_artifact_score"": 1,\n ""dataset_usefulness_score"": 5,...",NaN,0,15.0,5.000000,keep,total_score_15_ge_12_keep
4196,aya::human_031646_para_r2,human_031646,human_031646_para_r2,aya,fb,"Lại va phải ""nó"" rồi :) #HwangLong #welax","Thách thức ""nó"" luôn luôn thôi :) #HwangLong #viux",0.5714,False,False,False,False,False,False,judge,8.0,9,2,paraphrase,0.0950,0.7918,0.1410,27,0.3571,5.0,4.0,5.0,5.0,1.0,5.0,5.0,review,drop,Candidate has slight AI artifact (template-like phrasing) but preserves meaning and social style. Minor inconsistency in language use and slight naturalness issues.,"{\n ""meaning_score"": 5,\n ""naturalness_score"": 4,\n ""social_style_score"": 5,\n ""language_consistency_score"": 5,\n ""ai_artifact_score"": 1,\n ""dataset_usefulness_sco

## 9. Summary by generator


In [18]:

if "generator" not in df.columns:
    df["generator"] = "unknown"

summary = (
    df.groupby("generator")
    .agg(
        total=("judge_item_id", "count"),
        keep=("three_criteria_decision", lambda x: (x == "keep").sum()),
        review=("three_criteria_decision", lambda x: (x == "review").sum()),
        drop=("three_criteria_decision", lambda x: (x == "drop").sum()),
        keep_rate=("three_criteria_decision", lambda x: (x == "keep").mean()),
        review_rate=("three_criteria_decision", lambda x: (x == "review").mean()),
        drop_rate=("three_criteria_decision", lambda x: (x == "drop").mean()),
        avg_3criteria_total_score=("quality_3criteria_total_score", "mean"),
        avg_3criteria_quality_score=("quality_3criteria_avg_score", "mean"),
        avg_meaning=("meaning_score", "mean"),
        avg_naturalness=("naturalness_score", "mean"),
        avg_social_style=("social_style_score", "mean"),
        avg_missing_3criteria_score_count=("missing_3criteria_score_count", "mean"),
    )
    .reset_index()
)

if "final_decision" in df.columns:
    original_summary = (
        df.groupby("generator")
        .agg(
            original_keep=("final_decision", lambda x: (x.astype(str).str.lower() == "keep").sum()),
            original_review=("final_decision", lambda x: (x.astype(str).str.lower() == "review").sum()),
            original_drop=("final_decision", lambda x: (x.astype(str).str.lower() == "drop").sum()),
        )
        .reset_index()
    )
    summary = summary.merge(original_summary, on="generator", how="left")

summary["quality_index_3criteria"] = (
    0.50 * summary["keep_rate"]
    + 0.30 * (summary["avg_3criteria_total_score"] / MAX_TOTAL_SCORE)
    + 0.20 * (1 - summary["drop_rate"])
) * 100

def add_quality_level(score):
    if score >= 85:
        return "High"
    elif score >= 75:
        return "Good"
    elif score >= 65:
        return "Acceptable"
    else:
        return "Low"

summary["quality_level"] = summary["quality_index_3criteria"].apply(add_quality_level)
summary = summary.sort_values("quality_index_3criteria", ascending=False)

summary_output = output_dir / "summary_3criteria_by_generator.csv"
summary.to_csv(summary_output, index=False, encoding="utf-8-sig")

print(f"Summary saved to: {summary_output}")
display(summary)


Summary saved to: D:\IE403\ViAIGS\outputs\llm_judge_3criteria\summary_3criteria_by_generator.csv


,generator,total,keep,review,drop,keep_rate,review_rate,drop_rate,avg_3criteria_total_score,avg_3criteria_quality_score,avg_meaning,avg_naturalness,avg_social_style,avg_missing_3criteria_score_count,original_keep,original_review,original_drop,quality_index_3criteria,quality_level
3,gpt,2064,1141,21,902,0.552810,0.010174,0.437016,7.895833,2.631944,2.635174,2.457364,2.803295,1.302326,86,17,1961,54.691860,Low
0,aya,1700,811,59,830,0.477059,0.034706,0.488235,7.055882,2.351961,2.251765,2.221765,2.582353,1.392353,32,14,1654,48.200000,Low
2,gemma,1614,744,97,773,0.460967,0.060099,0.478934,7.037794,2.345931,2.156753,2.233581,2.647460,1.319703,7,3,1604,47.545229,Low
1,deepseek,1897,830,57,1010,0.437533,0.030047,0.532420,6.388508,2.129503,2.050079,1.980496,2.357934,1.543490,8,15,1874,44.005271,Low
4,sailor,1726,646,116,964,0.374276,0.067207,0.558517,5.993048,1.997683,1.795481,1.909038,2.288528,1.510429,5,6,1715,39.529548,Low


## 10. Export essential columns


In [19]:

essential_cols = [
    "judge_item_id",
    "generator",
    "source",
    "original_id",
    "candidate_id",
    "original_text",
    "candidate_text",
    "meaning_score",
    "naturalness_score",
    "social_style_score",
    "quality_3criteria_total_score",
    "quality_3criteria_avg_score",
    "three_criteria_decision",
    "three_criteria_reason",
]
essential_cols = [c for c in essential_cols if c in df.columns]

essential_df = df[essential_cols].copy()

essential_output = output_dir / "final_3criteria_essential.csv"
accepted_essential_output = output_dir / "accepted_3criteria_essential.csv"

essential_df.to_csv(essential_output, index=False, encoding="utf-8-sig")
essential_df[essential_df["three_criteria_decision"] == "keep"].to_csv(
    accepted_essential_output,
    index=False,
    encoding="utf-8-sig"
)

print(f"Essential full file: {essential_output}")
print(f"Essential accepted file: {accepted_essential_output}")
display(essential_df.head())


Essential full file: D:\IE403\ViAIGS\outputs\llm_judge_3criteria\final_3criteria_essential.csv
Essential accepted file: D:\IE403\ViAIGS\outputs\llm_judge_3criteria\accepted_3criteria_essential.csv


,judge_item_id,generator,source,original_id,candidate_id,original_text,candidate_text,meaning_score,naturalness_score,social_style_score,quality_3criteria_total_score,quality_3criteria_avg_score,three_criteria_decision,three_criteria_reason
0,aya::human_013660_para_r1,aya,fb,human_013660,human_013660_para_r1,Nguyễn Bích Như học tập đi,"Nâng niu mơ ước, Nguyễn Bích Như hãy cố gắng học tập thật tốt!",0.0,0.0,0.0,0.0,0.0,drop,total_score_0_le_6_drop
1,aya::human_025981_para_r3,aya,ytb,human_025981,human_025981_para_r3,Trên đất nước nghèo đói của Việt Nam có mấy người như anh Vũ. Người toàn tâm toàn ý vì sự phát triến đất nước. Nhiều người có chúc quyền cao sẽ đồ kỵ Anh. Vì anh không...,"Trên đất nước Việt Nam nghèo khó, có những cá nhân như anh Vũ, những con người cống hiến hết mình cho sự phát triển của đất nước. Nhiều người nắm quyền lực sẽ e ngại anh vì sự ...",0.0,0.0,0.0,0.0,0.0,drop,total_score_0_le_6_drop
2,aya::human_028119_para_r2,aya,ytb,human_028119,human_028119_para_r2,"Mình giới thiệu 1 cuốn sách trinh thám kinh điển của 1 tác giả Việt Nam, xuất bản năm 2000. Sách được chuyển thể thành phim, và gây tiếng vang lớn trong những năm đầu thế kỷ 21...","Cuốn sách trinh thám kinh điển, ra mắt vào năm 2000, của một tác giả người Việt Nam đã thu hút sự chú ý lớn trong những năm đầu thập kỷ 21 khi xã hội Việt Nam vẫn còn khá chưa ...",0.0,0.0,0.0,0.0,0.0,drop,total_score_0_le_6_drop
3,aya::human_016713_para_r3,aya,ytb,human_016713,human_016713_para_r3,Mua dép thì phải làm như thế nào ạ?,"Khi mua dép, bạn cần lưu ý những điều sau đây nhé!",0.0,0.0,0.0,0.0,0.0,drop,total_score_0_le_6_drop
4,aya::human_014594_para_r3,aya,fb,human_014594,human_014594_para_r3,Thảo Uyên Nguyễn,Thảo Uyên Nguyễn vừa bày tỏ tâm trạng mới nhất của cô ấy trên mạng xã hội.,0.0,0.0,0.0,0.0,0.0,drop,total_score_0_le_6_drop


## Final output

File dùng tiếp cho training/filtering:

```text
outputs/llm_judge_3criteria/accepted_3criteria.csv
```

File gọn hơn:

```text
outputs/llm_judge_3criteria/accepted_3criteria_essential.csv
```
